In [0]:

# Databricks notebook source

# Customer360 - Bronze Environment Setup

CATALOG = "dev_customer360"
BRONZE_SCHEMA = "bronze"

BASE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/landing"
OLIST_PATH = f"{BASE_PATH}/olist"

ENTITIES = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation"
]

print("Customer360 Bronze Setup")
print(f"Catalog: {CATALOG}")
print(f"Bronze schema: {BRONZE_SCHEMA}")
print(f"Landing path: {OLIST_PATH}")
print(f"Configured entities: {len(ENTITIES)}")


Customer360 Bronze Setup
Catalog: dev_customer360
Bronze schema: bronze
Landing path: /Volumes/dev_customer360/bronze/landing/olist
Configured entities: 9


In [0]:

# Confirm that the existing Unity Catalog
# resources are accessible.

spark.sql(
    f"DESCRIBE CATALOG {CATALOG}"
).display()

spark.sql(
    f"SHOW SCHEMAS IN {CATALOG}"
).display()

spark.sql(
    f"SHOW VOLUMES IN {CATALOG}.{BRONZE_SCHEMA}"
).display()


info_name,info_value
Catalog Name,dev_customer360
Comment,
Owner,karayatprabhakar@gmail.com
Catalog Type,Regular


databaseName
bronze
default
gold
information_schema
ops
silver


database,volume_name
bronze,landing


In [0]:

# Check the existing Olist landing directory.

dbutils.fs.ls(OLIST_PATH)

print("Bronze landing location accessible.")


Bronze landing location accessible.


In [0]:

# Check the source entity directories.

for entity in ENTITIES:
    entity_path = f"{OLIST_PATH}/{entity}/"

    try:
        files = dbutils.fs.ls(entity_path)

        print(
            f"OK: {entity:<22} "
            f"entries={len(files)}"
        )

    except Exception as exc:
        print(
            f"CHECK REQUIRED: {entity} | {exc}"
        )


OK: customers              entries=5
OK: orders                 entries=1
OK: order_items            entries=4
OK: payments               entries=1
OK: reviews                entries=1
OK: products               entries=1
OK: sellers                entries=1
OK: geolocation            entries=1
OK: category_translation   entries=1


In [0]:

import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config.schemas import SCHEMA_REGISTRY

print(
    f"Registered entities: "
    f"{len(SCHEMA_REGISTRY)}"
)

for entity_name in SCHEMA_REGISTRY:
    print(entity_name)


Registered entities: 9
customers
orders
order_items
payments
reviews
products
sellers
geolocation
category_translation


In [0]:

# Ensure every required entity has a schema.

missing_entities = (
    set(ENTITIES) - set(SCHEMA_REGISTRY.keys())
)

unexpected_entities = (
    set(SCHEMA_REGISTRY.keys()) - set(ENTITIES)
)

assert not missing_entities, (
    f"Missing schemas: {missing_entities}"
)

assert not unexpected_entities, (
    f"Unexpected schemas: {unexpected_entities}"
)

assert len(SCHEMA_REGISTRY) == 9

print(
    "PASSED: All 9 Olist entity schemas "
    "are registered."
)


PASSED: All 9 Olist entity schemas are registered.


In [0]:

# Verify Bronze ingestion targets.

expected_tables = [
    f"bronze_{entity}"
    for entity in ENTITIES
]

existing_tables = {
    row.tableName
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"
    ).collect()
}

missing_tables = []

for table_name in expected_tables:
    if table_name in existing_tables:
        print(f"OK: {table_name}")
    else:
        missing_tables.append(table_name)
        print(f"MISSING: {table_name}")

assert not missing_tables, (
    f"Missing Bronze tables: {missing_tables}"
)

print(
    "PASSED: All 9 Bronze tables exist."
)


OK: bronze_customers
OK: bronze_orders
OK: bronze_order_items
OK: bronze_payments
OK: bronze_reviews
OK: bronze_products
OK: bronze_sellers
OK: bronze_geolocation
OK: bronze_category_translation
PASSED: All 9 Bronze tables exist.


In [0]:

print("=" * 55)
print("CUSTOMER360 BRONZE ENVIRONMENT")
print("=" * 55)

print(f"Catalog          : {CATALOG}")
print(f"Schema           : {BRONZE_SCHEMA}")
print(f"Landing location : {OLIST_PATH}")
print(f"Source entities  : {len(ENTITIES)}")
print(f"Schema registry  : {len(SCHEMA_REGISTRY)}")
print(f"Bronze tables    : {len(expected_tables)}")

print("=" * 55)
print("BRONZE SETUP VERIFICATION COMPLETED")
print("=" * 55)


CUSTOMER360 BRONZE ENVIRONMENT
Catalog          : dev_customer360
Schema           : bronze
Landing location : /Volumes/dev_customer360/bronze/landing/olist
Source entities  : 9
Schema registry  : 9
Bronze tables    : 9
BRONZE SETUP VERIFICATION COMPLETED
